In [ ]:
import os
import langchain
import os
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
import glob
import pandas as pd

In [2]:
from datasets import Dataset
from ragas import evaluate
import ragas
import ragas.metrics as metrics

In [3]:
pdf_paths = glob.glob("pdfs/*.pdf")

In [4]:
docs = []

for path in pdf_paths:

    loader = PyPDFLoader(path)

    docs.extend(loader.load())

print("PDF sayısı:", len(pdf_paths))

print("Toplam sayfa:", len(docs))

for p in pdf_paths:

    print(p)

PDF sayısı: 6
Toplam sayfa: 36
pdfs/solar_energy_innovations.pdf
pdfs/battery_storage_and_sodium_ion_batteries.pdf
pdfs/wind_energy_technologies.pdf
pdfs/global_energy_transition_2026.pdf
pdfs/istanbul_micro_grid_case_study.pdf
pdfs/smart_grids_and_ai.pdf


In [5]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6552.71it/s]


In [6]:
splitters = {
    "recursive": RecursiveCharacterTextSplitter(

        chunk_size=1000,

        chunk_overlap=100

    ),

    "semantic": SemanticChunker(

        embeddings=embeddings

    )

}

### Comparing Text Splitting Methods

Apply each text splitter to the documents and store the resulting chunks for comparison. The number of chunks generated by each method is also reported.

In [8]:
splits_by_method = {}

for name, splitter in splitters.items():

    splits = splitter.split_documents(docs)

    splits_by_method[name] = splits

    print(name, "chunk count:", len(splits))

recursive chunk count: 79
semantic chunk count: 72


In [9]:
vectorstores = {}

for name, splits in splits_by_method.items():

    vectorstores[name] = FAISS.from_documents(splits, embeddings)

    print(name, "FAISS created")

recursive FAISS created
semantic FAISS created


In [10]:
question = "How does artificial intelligence improve smart grid management?"

for name, vectorstore in vectorstores.items():
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    docs_found = retriever.invoke(question)

    print("\n===================")
    print(name)
    print("===================")

    for i, doc in enumerate(docs_found):
        print(f"\nChunk {i+1}")
        print(doc.page_content[:500])


recursive

Chunk 1
smart meters, weather forecasts, market information, and operational equipment.
Traditional analytical methods often struggle to process such large datasets efficiently.
AI algorithms can identify patterns, make predictions, and optimize decisions at scales
that would be impossible through manual analysis.
Several AI techniques are commonly used within smart grid environments:
•Machine Learning
•Deep Learning
•Reinforcement Learning
•Neural Networks
•Predictive Analytics
These technologies suppo

Chunk 2
11 Conclusion
Smart grids represent a fundamental evolution in electricity infrastructure. By combin-
ing digital technologies, advanced communication systems, and automation capabilities,
smart grids enable more efficient, reliable, and sustainable energy management.
Artificial intelligence significantly enhances these capabilities by improving demand
forecasting, renewable energy integration, predictive maintenance, fault detection, and
cybersecurity. Together, sm

In [11]:
eval_questions = [
    {
        "question": "How does artificial intelligence improve smart grid management?",
        "ground_truth": "Artificial intelligence improves smart grid management by supporting demand forecasting, renewable energy integration, predictive maintenance, fault detection, cybersecurity, and real-time optimization."
    },
    {
        "question": "What are the main advantages of sodium-ion batteries?",
        "ground_truth": "Sodium-ion batteries offer lower raw material costs, greater resource availability, reduced supply chain risks, improved sustainability, and good low-temperature performance."
    },
    {
        "question": "How do Perovskite-Silicon tandem solar cells improve solar energy generation?",
        "ground_truth": "They combine a perovskite layer with a silicon base layer so that different parts of the solar spectrum can be absorbed more effectively, increasing efficiency compared with conventional silicon cells."
    },
    {
        "question": "Why is offshore wind energy important?",
        "ground_truth": "Offshore locations generally provide stronger and more consistent wind conditions, allowing higher electricity generation and larger turbine deployments."
    },
    {
        "question": "What is the purpose of battery storage in renewable energy systems?",
        "ground_truth": "Battery storage stores excess renewable electricity and releases it later when demand increases or renewable generation decreases."
    },
    {
        "question": "What are the main characteristics of smart grids?",
        "ground_truth": "Smart grids provide two-way communication, real-time monitoring, automated fault response, renewable integration, support for distributed energy resources, and improved reliability."
    },
    {
        "question": "What role do smart meters play in smart grids?",
        "ground_truth": "Smart meters automatically collect and transmit electricity consumption data while providing near real-time usage information."
    },
    {
        "question": "What are the main benefits of predictive maintenance in wind turbines?",
        "ground_truth": "Predictive maintenance identifies failures before they occur, reducing downtime, maintenance costs, and unexpected outages."
    },
    {
        "question": "How does AI improve wind turbine performance?",
        "ground_truth": "AI supports predictive maintenance, power forecasting, fault detection, and blade pitch optimization."
    },
    {
        "question": "What are the advantages of Building-Integrated Photovoltaics (BIPV)?",
        "ground_truth": "BIPV integrates solar generation directly into building materials such as roofs, facades, and windows while preserving functionality and aesthetics."
    },
    {
        "question": "Why are Perovskite solar cells considered promising?",
        "ground_truth": "Perovskite solar cells offer high efficiency, low manufacturing costs, lightweight structures, and flexible deployment options."
    },
    {
        "question": "What challenges do renewable energy systems create for electricity grids?",
        "ground_truth": "Renewable generation is variable and weather-dependent, creating challenges related to intermittency, balancing supply and demand, and maintaining grid stability."
    },
    {
        "question": "How do battery management systems use artificial intelligence?",
        "ground_truth": "AI analyzes battery parameters such as voltage, current, temperature, state of charge, and state of health to optimize operation and predict failures."
    },
    {
        "question": "What are the key benefits of offshore wind farms compared to onshore wind farms?",
        "ground_truth": "Offshore wind farms benefit from stronger and more consistent winds, larger turbines, and higher energy production."
    },
    {
        "question": "What is Vehicle-to-Grid (V2G) technology?",
        "ground_truth": "Vehicle-to-Grid technology allows electric vehicles to return stored electricity to the grid when needed."
    },
    {
        "question": "How does blockchain support the Istanbul Micro-Grid Project?",
        "ground_truth": "Blockchain enables transparent peer-to-peer energy trading through smart contracts and secure transaction records."
    },
    {
        "question": "What are Green Tokens in the Istanbul Micro-Grid Project?",
        "ground_truth": "Green Tokens are digital rewards provided for renewable energy generation and energy-saving activities."
    },
    {
        "question": "How do community battery systems support micro-grids?",
        "ground_truth": "Community battery systems store excess renewable energy, support peak demand periods, and provide backup power during outages."
    },
    {
        "question": "What are the major challenges facing sodium-ion battery technology?",
        "ground_truth": "The main challenge is lower energy density compared with advanced lithium-ion batteries."
    },
    {
        "question": "How does renewable energy contribute to the global energy transition?",
        "ground_truth": "Renewable energy reduces dependence on fossil fuels, lowers greenhouse gas emissions, and supports sustainable electricity generation."
    },
    {
    "question": "How do battery storage systems and smart grids work together to support renewable energy integration?",
    "ground_truth": "Battery storage systems store excess renewable electricity while smart grids use forecasting, monitoring, and optimization technologies to balance supply and demand."
},
{
    "question": "How are artificial intelligence technologies used across both wind farms and smart grids?",
    "ground_truth": "AI is used for predictive maintenance, forecasting, fault detection, optimization, and operational decision support in both wind farms and smart grids."
},
{
    "question": "How do renewable energy generation, battery storage, and micro-grids contribute to energy resilience?",
    "ground_truth": "Renewable generation provides local power, battery storage balances supply and demand, and micro-grids enable localized operation during grid disruptions."
}]

In [12]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model="llama3.2:1b")

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

If the answer is not in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{input}
""")

document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [13]:
rows = []

for splitter_name, vectorstore in vectorstores.items():
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    rag_chain = create_retrieval_chain(retriever, document_chain)

    for item in eval_questions:
        question = item["question"]

        retrieved_docs = retriever.invoke(question)
        contexts = [doc.page_content for doc in retrieved_docs]

        response = rag_chain.invoke({"input": question})
        answer = response["answer"]

        rows.append({
            "splitter": splitter_name,
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": item["ground_truth"]
        })

df = pd.DataFrame(rows)
df[["splitter", "question", "answer"]].head()

,splitter,question,answer
0,recursive,How does artificial intelligence improve smart...,Artificial intelligence improves smart grid ma...
1,recursive,What are the main advantages of sodium-ion bat...,The main advantages of sodium-ion batteries me...
2,recursive,How do Perovskite-Silicon tandem solar cells i...,Perovskite-Silicon Tandem Solar Cells improve ...
3,recursive,Why is offshore wind energy important?,Offshore wind energy is important due to its a...
4,recursive,What is the purpose of battery storage in rene...,Battery storage in renewable energy systems se...


In [14]:
df[["splitter", "question", "answer"]].head()

,splitter,question,answer
0,recursive,How does artificial intelligence improve smart...,Artificial intelligence improves smart grid ma...
1,recursive,What are the main advantages of sodium-ion bat...,The main advantages of sodium-ion batteries me...
2,recursive,How do Perovskite-Silicon tandem solar cells i...,Perovskite-Silicon Tandem Solar Cells improve ...
3,recursive,Why is offshore wind energy important?,Offshore wind energy is important due to its a...
4,recursive,What is the purpose of battery storage in rene...,Battery storage in renewable energy systems se...


In [15]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
evaluator_llm = LangchainLLMWrapper(
    ChatOllama(model="qwen2.5:3b",
               temperature=0,
               format="json")
)

evaluator_embeddings = LangchainEmbeddingsWrapper(

    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7579.05it/s]


In [17]:
from ragas.run_config import RunConfig

run_config = RunConfig(

    timeout=600,
    max_retries=1)

In [18]:
row = df.iloc[0]

print(len(row["contexts"]))

print(sum(len(c) for c in row["contexts"]))

4
3392


In [19]:
df.iloc[0]

splitter                                                recursive
question        How does artificial intelligence improve smart...
answer          Artificial intelligence improves smart grid ma...
contexts        [smart meters, weather forecasts, market infor...
ground_truth    Artificial intelligence improves smart grid ma...
Name: 0, dtype: object

In [ ]:
small_dataset = Dataset.from_pandas(
    df.head(1)[["question", "answer", "contexts", "ground_truth"]]
)

result = evaluate(
    small_dataset,
    metrics=[faithfulness],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
    run_config=run_config
)

print(result)

Evaluating: 100%|██████████| 1/1 [01:07<00:00, 67.99s/it]

{'faithfulness': 1.0000}


In [22]:
for splitter in ["recursive", "semantic"]:

    splitter_df = df[df["splitter"] == splitter]

    ragas_dataset = Dataset.from_pandas(
        splitter_df[["question", "answer", "contexts", "ground_truth"]]
    )

    result = evaluate(
        ragas_dataset,
        metrics=[
            answer_relevancy,
            context_precision,
            context_recall
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False,
        run_config=run_config
    )

    print("\n", splitter)
    print(result)

Evaluating: 100%|██████████| 69/69 [17:02<00:00, 14.82s/it]



 recursive
{'answer_relevancy': 0.8494, 'context_precision': 0.8249, 'context_recall': 0.9565}


Evaluating: 100%|██████████| 69/69 [17:02<00:00, 14.82s/it]


 semantic
{'answer_relevancy': 0.7813, 'context_precision': 0.7440, 'context_recall': 0.9239}
